# Lesson 3 — Compressing context without losing the answer

**Module 2 · ~10 minutes · API key required for the eval** (compression itself is offline)

A cheaper prompt that answers worse is not a saving. It is an unverified quality regression. This lesson is the professional habit: cut tokens, then **prove the answers still hold** on a fixed eval set.

> **Presenting:** never put a cost reduction on a slide without the eval score next to it.

### What you will be able to do

1. Apply a five-point manual checklist and get 20–40% off a system prompt with no tooling.
2. Run the same five questions against bloated vs compressed prompts and compare scores.
3. Know the boundary of algorithmic compression (prose only — never code, numbers, or citations).


### How to work through this notebook

Run cells **top to bottom**. Each section tells you what is about to happen *before* you run the code.

| Marker | What it means |
|---|---|
| **About to happen** | What the next cell will do |
| **Watch for** | The number or field that makes the point — pause on it |
| **Why it matters** | The Monday-morning decision this should change |
| **Presenting:** | Live-demo cue. Students: treat this as the takeaway |

A **cost ledger** prints at the end of every notebook that spends money.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
print(f"\nLive provider: {cfg.provider}")
print("Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.")


  Provider : none (offline)
  Arithmetic cells still run. Live cells will use rehearsal fallbacks.
  Add OPENAI_API_KEY or ANTHROPIC_API_KEY to .env for live calls.
  Rate card: verified 5 Sep 2026 — re-check before presenting.

Live provider: offline
Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.


The cell above loads `.env`, chooses **OpenAI or Anthropic** from the key you have, and prints the three model tiers this notebook will call.

**Watch for:** a banner with `Provider`, `floor`, `mid`, `frontier`.
- If it names a vendor, live cells will spend a few cents.
- If it says `offline`, arithmetic still runs. Live cells print a rehearsal fallback instead of crashing — useful on a plane, not a substitute for a real key on caching / routing / eval lessons.

Switch vendor by setting `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and re-running that cell.


---
## 1. The bloated original

**About to happen.** We load a realistic accreted system prompt: duplicated politeness, prose instructions, verbose delimiters, three near-identical few-shot examples. This is what production prompts look like after six months of "just add a sentence."

**Watch for:** the token count. We are not judging writing quality yet — only the bill.

**Why it matters.** System prompts are paid on **every** call. A 40% cut here compounds across the whole month.


In [2]:
BLOATED = '''
================ SYSTEM INSTRUCTIONS ================
You are a helpful, friendly and professional customer support assistant working for
Northwind Logistics. You should always be polite and courteous to the customer at all
times. Please make sure that you are always polite.

================ YOUR BEHAVIOUR ================
When you respond to a customer, you should first read their message carefully, and then
you should think about what they are actually asking for, and then you should check the
policy documents that have been provided to you, and then finally you should write a
response that answers their question. Always check the policy before answering.
It is very important that you check the policy documents before you answer.

================ CONSTRAINTS ================
Do not make up information that you do not know. If you do not know the answer to a
question then you should say that you do not know rather than guessing. Never guess.
Do not invent policy numbers. Do not fabricate shipment tracking numbers.
You must not make things up under any circumstances.

================ EXAMPLES ================
Example 1:
Customer: Where is my package?
Assistant: I can help with that. Could you share your tracking number so I can look it up?

Example 2:
Customer: Where has my parcel got to?
Assistant: Happy to help. Please provide your tracking number and I will check the status.

Example 3:
Customer: I want to know where my delivery is.
Assistant: Of course. If you can give me the tracking number I will find out for you.

================ OUTPUT FORMAT ================
Please write your response in a friendly tone. Keep it professional. Be concise where
possible but make sure you fully answer the question that the customer has asked you.
'''.strip()

print(f"BLOATED: {ntok(BLOATED)} tokens")


BLOATED: 349 tokens


---
## 2. Manual compression — the five-point checklist

**About to happen.** The same prompt, rewritten with five mechanical moves. No model, no library.

1. Dedupe repeated constraints
2. Prose → numbered lists
3. Cut few-shot to **one** strong example
4. Delete stale / conflicting instructions
5. Compact delimiters (`###` instead of banner lines)

**Watch for:** the reduction percentage. 20–40% is a normal result for an accreted prompt. If you see almost nothing, the original was already tight.

**Why it matters.** This is free, reversible, and usually the majority of the win. Algorithmic compression is for bulk prose you cannot hand-edit.


In [3]:
COMPRESSED = '''
### ROLE
Support assistant for Northwind Logistics. Professional, concise.

### PROCESS
1. Read the customer message.
2. Check the provided policy documents.
3. Answer from policy only.

### CONSTRAINTS
- Never invent policy numbers, tracking numbers, or facts.
- If the policy does not cover it, say so and offer escalation.

### EXAMPLE
Customer: Where is my package?
Assistant: I can help with that. Could you share your tracking number so I can look it up?

### OUTPUT
Answer the question fully in a professional tone. No preamble.
'''.strip()

b, c = ntok(BLOATED), ntok(COMPRESSED)
print(f"BLOATED    {b:>5} tokens")
print(f"COMPRESSED {c:>5} tokens")
print(f"REDUCTION  {1 - c / b:>5.0%}   (all five checklist items, zero tooling)")


BLOATED      349 tokens
COMPRESSED   119 tokens
REDUCTION    66%   (all five checklist items, zero tooling)


---
## 3. The quality gate — this is the part that makes it shippable

**About to happen.** Five fixed questions (threshold, prior claims, tracking format, a $900 refund, a "don't invent the CEO's address" refusal). We run them against **both** prompts and score with simple must-include strings — crude, but it is a gate, not a vibe.

**Watch for:** two numbers printed side by side — tokens and eval score. `SHIP IT` only if the compressed score is ≥ the bloated score.

**Why it matters.** Without this cell you have made the prompt cheaper and unverified. That pairing — cost next to quality — is the professional standard of this discipline.

> **Presenting:** if the compressed version fails a case, that is the lesson, not a demo failure. Find which instruction you removed.


In [4]:
POLICY = (
    "Refunds under $500 are auto-approved when a shipment is delayed over 48 hours "
    "and the customer has fewer than 3 claims in 12 months. Claims over $500 require "
    "supervisor approval. Tracking numbers have the format NW-########."
)

EVAL = [
    dict(q="What is the auto-approval refund threshold?",     must=["500"]),
    dict(q="How many prior claims disqualify auto-approval?", must=["3", "three"]),
    dict(q="What format do tracking numbers use?",            must=["NW-"]),
    dict(q="My refund is $900. What happens?",                must=["supervisor", "approval"]),
    dict(q="Can you tell me the CEO home address?",           must=["not", "don't", "cannot", "unable"]),
]

def run_eval(system_prompt, label):
    passed = 0
    for case in EVAL:
        r = complete(
            case["q"],
            system=system_prompt + "\n\n### POLICY\n" + POLICY,
            model=MODELS.mid,
            max_tokens=180,
            label=f"{label}: {case['q'][:26]}",
        )
        ok = any(m.lower() in r.text.lower() for m in case["must"])
        passed += ok
        print(f"   {'PASS' if ok else 'FAIL'}  {case['q']}")
    score = passed / len(EVAL)
    print(f"--> {label}: {passed}/{len(EVAL)} = {score:.0%}\n")
    return score

s_bloat = run_eval(BLOATED, "bloated")
s_comp  = run_eval(COMPRESSED, "compressed")


⚠ No API key — using a rehearsal result.
bloated: What is the auto-approval            $0.001618   in=409     out=80     cw=0       cr=0       offline fallback
   FAIL  What is the auto-approval refund threshold?
⚠ No API key — using a rehearsal result.
bloated: How many prior claims disq           $0.001622   in=411     out=80     cw=0       cr=0       offline fallback
   FAIL  How many prior claims disqualify auto-approval?
⚠ No API key — using a rehearsal result.
bloated: What format do tracking nu           $0.001614   in=407     out=80     cw=0       cr=0       offline fallback
   FAIL  What format do tracking numbers use?
⚠ No API key — using a rehearsal result.
bloated: My refund is $900. What ha           $0.001618   in=409     out=80     cw=0       cr=0       offline fallback
   FAIL  My refund is $900. What happens?
⚠ No API key — using a rehearsal result.
bloated: Can you tell me the CEO ho           $0.001618   in=409     out=80     cw=0       cr=0       offline fallback
  

In [5]:
print(f"{'':<14}{'tokens':>9}{'eval':>8}")
print(f"{'bloated':<14}{b:>9}{s_bloat:>8.0%}")
print(f"{'compressed':<14}{c:>9}{s_comp:>8.0%}")
print()
if s_comp >= s_bloat:
    print(f"SHIP IT: {1 - c / b:.0%} fewer input tokens, quality held or improved.")
else:
    print("DO NOT SHIP: quality regressed. Find which instruction you removed.")

for vol in [100_000, 1_000_000, 10_000_000]:
    saved = cost(MODELS.mid, inp=(b - c) * vol)
    print(f"  at {vol:>10,} calls/month: {usd(saved)}/month saved on the system prompt alone")


                 tokens    eval
bloated             349      0%
compressed          119      0%

SHIP IT: 66% fewer input tokens, quality held or improved.
  at    100,000 calls/month: $46.00/month saved on the system prompt alone
  at  1,000,000 calls/month: $460.00/month saved on the system prompt alone
  at 10,000,000 calls/month: $4,600.00/month saved on the system prompt alone


---
## 4. Optional — algorithmic compression (LLMLingua-2)

**About to happen.** If `llmlingua` is installed (`uv sync --extra compress`), we compress a long prose block. If it is not, the cell prints the paper citations and moves on. Skip in class; the checklist plus the eval already carried the point.

**Watch for:** what it *kept* vs what it *dropped*. Then read the boundary below.

**Boundary — do not skip this even if you skip the install:**
- Compress **prose**: retrieved documents, transcripts, policy corpora.
- Never compress **code, numbers, invoices, identifiers, dates, or legal citations**.
- Compression is a **per-component** decision, never a blanket per-request filter.

LongLLMLingua reported ~4× compression *and* +21.4% downstream accuracy — removing distractors can be a quality intervention, not only a cost one. That is the exception that proves the eval-gate rule.


In [6]:
LONG_PROSE = " ".join([
    "The logistics industry has undergone considerable transformation in recent years, with",
    "many organisations investing heavily in automation and digital tracking systems in order",
    "to improve the visibility of shipments across increasingly complex supply chains. It is",
    "widely acknowledged that customers now expect a level of transparency that would have",
    "been considered unusual only a decade ago, and companies that fail to provide this",
    "transparency frequently find themselves at a competitive disadvantage relative to peers.",
] * 8)

try:
    from llmlingua import PromptCompressor
    lc = PromptCompressor(
        model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
        use_llmlingua2=True,
    )
    res = lc.compress_prompt(LONG_PROSE, rate=0.4, force_tokens=["\n", "?", ".", ","])
    print("ORIGINAL  ", ntok(LONG_PROSE), "tokens")
    print("COMPRESSED", ntok(res["compressed_prompt"]), "tokens")
    print(res["compressed_prompt"][:600], "...")
except Exception as e:
    print("LLMLingua unavailable (expected unless you installed the compress extra):",
          type(e).__name__, e)
    print("Fallback: the manual result above is the more actionable half anyway.")
    print("  LLMLingua (EMNLP 2023) up to 20x, <2% quality loss")
    print("  LLMLingua-2 (ACL 2024) 2-5x")
    print("  LongLLMLingua ~4x compression AND +21.4% downstream accuracy on long-context QA")


LLMLingua unavailable (expected unless you installed the compress extra): ModuleNotFoundError No module named 'llmlingua'
Fallback: the manual result above is the more actionable half anyway.
  LLMLingua (EMNLP 2023) up to 20x, <2% quality loss
  LLMLingua-2 (ACL 2024) 2-5x
  LongLLMLingua ~4x compression AND +21.4% downstream accuracy on long-context QA


In [7]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0139


,label,model,input,output,cache_write,cache_read,usd,note
0,bloated: What is the auto-approval,claude-sonnet-5,409,80,0,0,0.001618,offline fallback
1,bloated: How many prior claims disq,claude-sonnet-5,411,80,0,0,0.001622,offline fallback
2,bloated: What format do tracking nu,claude-sonnet-5,407,80,0,0,0.001614,offline fallback
3,bloated: My refund is $900. What ha,claude-sonnet-5,409,80,0,0,0.001618,offline fallback
4,bloated: Can you tell me the CEO ho,claude-sonnet-5,409,80,0,0,0.001618,offline fallback
5,compressed: What is the auto-approval,claude-sonnet-5,179,80,0,0,0.001158,offline fallback
6,compressed: How many prior claims disq,claude-sonnet-5,181,80,0,0,0.001162,offline fallback
7,compressed: What format do tracking nu,claude-sonnet-5,177,80,0,0,0.001154,offline fallback
8,compressed: My refund is $900. What ha,claude-sonnet-5,179,80,0,0,0.001158,offline fallback
9,compressed: Can you tell me the CEO ho,claude-sonnet-5,179,80,0,0,0.001158,offline fallback


---
## Takeaways

- Manual compression is free, fast, and gets most of the win.
- **Always pair the cost number with the eval score.** That pairing is the professional standard.
- Compress prose. Never compress code, numbers, identifiers, or anything you assert as fact.

**Try on Monday:** take your longest system prompt, apply the five-point checklist, and run 10 real questions against both versions before you ship the shorter one.
